# 🧠 RAG Knowledge Agent — Retrieval + Real LLM Answering (Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/06_rag_knowledge_agent.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **What this does:** embeds your text with `all-MiniLM-L6-v2`, stores it in ChromaDB, retrieves the
> top-k chunks, and then **calls an OpenAI-compatible `/v1/chat/completions` endpoint** to produce the
> answer. Point it at the LLM server from
> [`01_t4_gpu_llm_api.ipynb`](01_t4_gpu_llm_api.ipynb) (or any Ollama/OpenAI endpoint).

- **Components:** `sentence-transformers` · `ChromaDB` · `requests` (no LangChain)
- **VRAM:** ~1.2 GB for embeddings (CPU works too)
- If no endpoint is configured the notebook still shows the retrieved context — it will **not** pretend
  to have generated an answer.
- The Gradio link is password protected (Basic auth).


In [ ]:
#@title 🚀 1. Hardware Check & Session Fingerprint
import hashlib, os, socket, time, uuid

os.environ["TOKENIZERS_PARALLELISM"] = "false"
raw_fp = f"colab-rag-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-rag-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique RAG Instance Fingerprint: {INSTANCE_ID}")


In [ ]:
#@title 📦 2. Install Vector Store & Embeddings
!pip install -q chromadb sentence-transformers pypdf gradio requests
print("✅ ChromaDB, sentence-transformers, PyPDF, Gradio installed.")


In [ ]:
#@title 🔍 3. Build the RAG Engine
import os
import time

import chromadb
import gradio as gr
import requests
from sentence_transformers import SentenceTransformer

print("⚡ Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.get_or_create_collection("rag_knowledge_base")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120


def chunk_text(text: str):
    """Simple character chunker with overlap; keeps retrieval predictable."""
    text = " ".join((text or "").split())
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end == len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return chunks


def index_document(title: str, content: str):
    chunks = chunk_text(content)
    if not chunks:
        return "❌ Nothing to index — paste some text first."
    stamp = int(time.time())
    collection.add(
        ids=[f"{title or 'doc'}-{stamp}-{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embedder.encode(chunks).tolist(),
        metadatas=[{"source": title or "untitled"} for _ in chunks],
    )
    return f"✅ Indexed {len(chunks)} chunks from '{title or 'untitled'}'. Collection size: {collection.count()}."


def retrieve(question: str, top_k: int):
    if collection.count() == 0:
        return []
    result = collection.query(
        query_embeddings=embedder.encode([question]).tolist(),
        n_results=min(int(top_k), collection.count()),
    )
    return result.get("documents", [[]])[0]


def query_knowledge(question: str, top_k: int, base_url: str, model: str, api_key: str):
    if not (question or "").strip():
        raise gr.Error("Please enter a question.")
    chunks = retrieve(question, top_k)
    if not chunks:
        return "⚠️ The index is empty — index a document first.", ""
    context = "\n\n---\n\n".join(chunks)

    if not (base_url or "").strip() or not (model or "").strip():
        return ("⚠️ No LLM endpoint configured. Add a Base URL (e.g. `http://localhost:11434/v1`) "
                "and model name to get a generated answer. Retrieved context is shown below."), context

    payload = {
        "model": model.strip(),
        "temperature": 0.2,
        "stream": False,
        "messages": [
            {"role": "system",
             "content": "Answer the question using ONLY the provided context. "
                        "If the context does not contain the answer, say so explicitly."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    }
    try:
        response = requests.post(
            f"{base_url.strip().rstrip('/')}/chat/completions",
            headers={"Authorization": f"Bearer {api_key or 'ollama'}"},
            json=payload,
            timeout=180,
        )
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer, context
    except Exception as exc:
        return f"❌ LLM request failed: {exc}", context


def studio_auth():
    """Password for the public Gradio link (Basic auth)."""
    import os
    import secrets

    password = os.getenv("STUDIO_PASSWORD", "")
    if not password:
        try:
            from google.colab import userdata
            password = userdata.get("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        try:
            from kaggle_secrets import UserSecretsClient
            password = UserSecretsClient().get_secret("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        password = secrets.token_urlsafe(9)
        print(f"\n🔑 One-time password for this session: {password}\n")
    return ("admin", password)

with gr.Blocks(title="🧠 RAG Vector Knowledge Studio") as demo:
    gr.Markdown("# 🧠 RAG Knowledge Studio\nEmbed. Retrieve. Answer with a real LLM.")
    with gr.Tab("📥 Ingest"):
        doc_title = gr.Textbox(label="Document title", value="System Architecture Guide")
        doc_content = gr.Textbox(label="Document text / Markdown / code", lines=8)
        ingest_btn = gr.Button("Index document", variant="primary")
        ingest_status = gr.Textbox(label="Status")
        ingest_btn.click(index_document, [doc_title, doc_content], [ingest_status])
    with gr.Tab("🔍 Ask"):
        question = gr.Textbox(label="Question", placeholder="What is the architecture of the system?")
        top_k = gr.Slider(1, 6, value=3, step=1, label="Top-K chunks")
        with gr.Accordion("LLM endpoint (OpenAI-compatible)", open=True):
            base_url = gr.Textbox(label="Base URL", value="http://localhost:11434/v1")
            model_name = gr.Textbox(label="Model", value="qwen2.5-coder:7b")
            api_key = gr.Textbox(label="API key (use your tunnel password)", type="password", value="")
        ask_btn = gr.Button("Retrieve + answer", variant="primary")
        answer = gr.Markdown(label="Answer")
        context_view = gr.Textbox(label="Retrieved context", lines=8)
        ask_btn.click(query_knowledge, [question, top_k, base_url, model_name, api_key],
                      [answer, context_view])

demo.launch(share=True, auth=studio_auth(), debug=False)
